Author: Konstantin Malanchev and the LINCC Frameworks team

Last updated: November 2026

# Light-curve features of known variable stars in Rubin DP2

In this notebook we'll show how to:
- Load the Rubin DP2 DiaObject catalog with nested light curves of forced photometry at the DiaObject positions
- Cross-match it with the VSX catalog of known variable stars and assign variability classes
- Filter light curves with quality flags and extract light-curve features in parallel with the `light-curve` package ([docs](https://light-curve.snad.space))
- Save the features to a Parquet file, which can be shared with non-DRHs
- Train a simple classifier to separate eclipsing binaries from LPVs

**Use case:** a DRH extracts light-curve features of known variable stars from Rubin data and shares them, so non-DRHs can develop and train machine-learning models without access to the Rubin data. The last part of this notebook uses the shared features only.

> [!NOTE]
> This notebook uses Rubin DP2 data, so it can be run by Rubin data rights holders (DRHs) only, e.g. on the [Rubin Science Platform](https://data.lsst.cloud) with the latest weekly image.
> Since we would like to share the notebook and its outputs with everyone, we never print or plot individual rows of the DP2 catalog: all the outputs are either catalog metadata or aggregated statistics.

In [ ]:
# lsdb and lsst-sphgeom are pre-installed on RSP, uncomment the line below if you are on another platform
# %pip install -qq lsdb lsst-sphgeom
# lsdb-rubin and light-curve are not pre-installed on RSP
%pip install -qq lsdb-rubin light-curve

In [ ]:
# Standard library imports
import logging
import os
import warnings

# Third-party imports
import astropy.units as u
import dask
import hats
import light_curve as licu
import lsdb
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
from astropy.coordinates import SkyCoord
from astropy.visualization.wcsaxes import Quadrangle
from dask.distributed import Client
from matplotlib import rcParams
from matplotlib.colors import LogNorm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Jupyter magic commands
%matplotlib inline

# Configuration
warnings.simplefilter("ignore")
logging.getLogger("distributed").setLevel(logging.WARNING)
rcParams["savefig.dpi"] = 150
rcParams["font.size"] = 14
plt.rc("font", family="serif")
mpl.rcParams["axes.linewidth"] = 1.5
dask.config.set({"dataframe.convert-string": False})

## Setup Dask cluster

Light curves need more memory than the object tables in the previous notebooks, so we use four workers with one thread each. By default, Dask splits the available memory between the workers, so each of them gets 8 GB of the 32 GB "Large" RSP container. The container has 16 CPUs, so each worker will run `light-curve` with four threads. As before, we use `/deleted-sundays/` for spilling data to disk.

In [ ]:
client = Client(
    n_workers=4,
    threads_per_worker=1,
    local_directory=f"/deleted-sundays/{os.environ.get('USER', 'dask_scratch')}",
)

# Print the dashboard link and port
print(f"Dask is running at: {client.dashboard_link}")

## Select a region on the sky

We need many known variable stars, so we look at the Galactic bulge: a 10×6 degree rectangle aligned with the Galactic equator and centered at the Galactic center. We define it by its corners in Galactic coordinates, convert them to RA and Dec, and use `PolygonSearch`, which connects the corners with great circles, so the region is only approximately a rectangle in Galactic coordinates. We also define a function to draw the rectangle on the sky density maps below, which we make with `hats.inspection.plot_density()`.

In [ ]:
# Galactic longitude and latitude ranges, in degrees
L_RANGE = (-5.0, 5.0)
B_RANGE = (-3.0, 3.0)

corners = SkyCoord(
    l=[L_RANGE[0], L_RANGE[1], L_RANGE[1], L_RANGE[0]] * u.deg,
    b=[B_RANGE[0], B_RANGE[0], B_RANGE[1], B_RANGE[1]] * u.deg,
    frame="galactic",
).icrs
polygon_search = lsdb.PolygonSearch(list(zip(corners.ra.deg, corners.dec.deg)))


def plot_rectangle(ax):
    """Draw the rectangle in red on a sky map made by LSDB."""
    rectangle = Quadrangle(
        (L_RANGE[0], B_RANGE[0]) * u.deg,
        (L_RANGE[1] - L_RANGE[0]) * u.deg,
        (B_RANGE[1] - B_RANGE[0]) * u.deg,
        edgecolor="red",
        facecolor="none",
        lw=2,
        transform=ax.get_transform("galactic"),
    )
    ax.add_patch(rectangle)

## Known variable stars from VSX

The [International Variable Star Index (VSX)](https://www.aavso.org/vsx/) is a public catalog of known variable stars, available in HATS format at [data.lsdb.io](https://data.lsdb.io). We use its variability types to make two labels, eclipsing binaries and long-period variables (LPVs), with regular expressions applied with the pandas `.str` accessor. A VSX type may combine several types, so an object may get both labels: we keep the objects which have exactly one of them.

In [ ]:
vsx = lsdb.open_catalog("https://data.lsdb.io/hats/vsx", columns=["Name", "Type"])

fig, ax = hats.inspection.plot_density(vsx.hc_structure, plot_title="VSX sky density", norm=LogNorm())
plot_rectangle(ax)
plt.show()

VSX covers the whole sky, so we select our rectangle with `.search()`, which is lazy as everything else:

In [ ]:
vsx = vsx.search(polygon_search)
vsx

In [ ]:
# A VSX type may combine several types, e.g. "EA+SR", so we look for each class in every component
ECLIPSING = r"(?:^|[+|])E[ABCW]?(?:[^A-Z]|$)"
LPV = r"(?:^|[+|])(?:M|SR[ABCDS]?|L[BC]?|LPV)(?:[^A-Z]|$)"


def add_labels(df):
    """Add is_eclipsing and is_lpv labels."""
    df["is_eclipsing"] = df["Type"].str.contains(ECLIPSING, na=False)
    df["is_lpv"] = df["Type"].str.contains(LPV, na=False)
    return df


vsx = vsx.map_partitions(add_labels)
# Keep objects which are either eclipsing binaries or LPVs, but not both
vsx = vsx.query("is_eclipsing != is_lpv")

VSX is public, so we can look at the data and count eclipsing binaries and LPVs in the field:

In [ ]:
vsx.head()

In [ ]:
vsx["is_eclipsing"].value_counts().compute()

## Open the DP2 DiaObject catalog

DiaObjects are objects detected on difference images. Each DiaObject has two nested light curves, stored with [nested-pandas](https://nested-pandas.readthedocs.io):
- `diaSource`: detections on the difference images;
- `diaObjectForcedSource`: forced photometry at the DiaObject position on every visit, measured on the direct images.

We use the forced photometry, because it gives a measurement for every visit, not only when the object was detected. We use the difference-image fluxes, `psfDiffFlux`, which measure the change of the object brightness relative to the template image, and for the uncertainties we use `psfDiffFluxErr_corrected`: errors corrected by a model trained to make the reduced χ² of non-variable objects close to unity, see the [DP2 page on data.lsdb.io](https://data.lsdb.io/Rubin/DP2/dia_object_collection) for details. We also load quality flags, to filter out bad observations.

In [ ]:
# DP2 DiaObject collection on RSP, check https://lsdb.io/dp2 for the paths on other platforms
dia_object = lsdb.open_catalog(
    "/rubin/lsdb_data/dp2/dia_object_collection",
    # coordinates and _healpix_29 are always loaded
    columns=[
        "diaObjectId",
        "diaObjectForcedSource.band",
        "diaObjectForcedSource.midpointMjdTai",
        "diaObjectForcedSource.psfDiffFlux",
        "diaObjectForcedSource.psfDiffFluxErr_corrected",
        "diaObjectForcedSource.psfDiffFlux_flag",
        "diaObjectForcedSource.psfDiffFluxErr_corrected_flag",
        "diaObjectForcedSource.diff_PixelFlags_nodataCenter",
        "diaObjectForcedSource.pixelFlags_saturatedCenter",
        "diaObjectForcedSource.invalidPsfFlag",
    ],
)

fig, ax = hats.inspection.plot_density(
    dia_object.hc_structure, plot_title="DP2 DiaObject sky density", norm=LogNorm()
)
plot_rectangle(ax)
plt.show()

Only a part of the rectangle is covered by DP2, and we select it with `.search()`:

In [ ]:
dia_object = dia_object.search(polygon_search)
dia_object

## Cross-match

We filter first and run expensive computations later, on a smaller sample: cross-matching with VSX leaves us with a small fraction of the DiaObjects. We use VSX as the left catalog, so the result is indexed by the public VSX coordinates.

In [ ]:
xmatch = vsx.crossmatch(
    dia_object, radius_arcsec=1.0, suffix_method="overlapping_columns", suffixes=("_vsx", "_dp2")
)
xmatch

## Filter the light curves

`query()` understands the `nested_column.sub_column` syntax, so we can filter individual observations: we keep g, r, i, z observations with valid fluxes and errors, and without any of the quality flags set. Then we keep objects with at least 20 observations left.

In [ ]:
BANDS = ["g", "r", "i", "z"]

xmatch = xmatch.query(
    "diaObjectForcedSource.band in ['g', 'r', 'i', 'z']"
    " & diaObjectForcedSource.psfDiffFlux.notna()"
    " & diaObjectForcedSource.psfDiffFluxErr_corrected.notna()"
    " & ~diaObjectForcedSource.psfDiffFlux_flag"
    " & ~diaObjectForcedSource.psfDiffFluxErr_corrected_flag"
    " & ~diaObjectForcedSource.diff_PixelFlags_nodataCenter"
    " & ~diaObjectForcedSource.pixelFlags_saturatedCenter"
    " & ~diaObjectForcedSource.invalidPsfFlag"
)
xmatch = xmatch.query("diaObjectForcedSource.len() >= 20")

`light-curve` requires observations to be sorted by time, and all the light-curve arrays to have the same data type. Fluxes are stored as `float32`, so we replace MJD with a `float32` time column, shifted by 60 000 days to keep the precision.

In [ ]:
def prepare_light_curves(df):
    """Sort observations by time and replace MJD with a float32 time column."""
    df = df.sort_values("diaObjectForcedSource.midpointMjdTai")
    df["diaObjectForcedSource.time"] = (df["diaObjectForcedSource.midpointMjdTai"] - 60_000).astype(
        np.float32
    )
    return df.drop(columns=["diaObjectForcedSource.midpointMjdTai"])


xmatch = xmatch.map_partitions(prepare_light_curves)

## Light-curve features

We'd like to share the features with non-DRHs, so we use features which describe the variability of the objects, but not their brightness. With `bands=...` each feature is evaluated for each passband independently, missing passbands give `NaN`. `OtsuSplit` splits the observations into two groups by flux, which may help to find eclipses. Some features, like the interpercentile range and the Otsu mean difference and standard deviations, are in flux units: we'll turn them into dimensionless features later, when training the classifier.

In [ ]:
extractor = licu.Extractor(
    licu.BeyondNStd(1.0, bands=BANDS),
    licu.Chi2Pvar(bands=BANDS),
    licu.InterPercentileRange(0.05, bands=BANDS),
    licu.Kurtosis(bands=BANDS),
    licu.OtsuSplit(bands=BANDS),
    licu.Skew(bands=BANDS),
    licu.StetsonK(bands=BANDS),
)
extractor.names

We apply the extractor to each partition with `map_partitions()`, following the [batch processing example](https://light-curve.snad.space/latest/features/batch_processing/) from the light-curve documentation: `Extractor.many()` takes the nested light curves as an Arrow array, without copying them into Python objects.

In [ ]:
def extract_features(df):
    """Extract light-curve features for a single partition."""
    result = df[["Name", "Type", "is_eclipsing", "diaObjectId"]]
    # Extractor.many() doesn't accept an empty list of light curves
    if len(df) == 0:
        return result.assign(**{name: np.array([], dtype=np.float32) for name in extractor.names})
    features = extractor.many(
        pa.array(df["diaObjectForcedSource"]),
        arrow_fields={"t": "time", "m": "psfDiffFlux", "sigma": "psfDiffFluxErr_corrected", "band": "band"},
        fill_value=np.nan,
        # 4 workers x 4 threads = 16 CPUs
        n_jobs=4,
    )
    return result.assign(**dict(zip(extractor.names, features.T)))


features = xmatch.map_partitions(extract_features)
features

## Save the features

The features catalog contains VSX names, types and labels, DiaObject IDs, and features. It is indexed by `_healpix_29` of the public VSX coordinates, but we don't need it, so we drop it. We save the features as plain Parquet files with `to_parquet()`, this runs the whole pipeline. It doesn't show a progress bar, so we recommend following the progress on the Dask dashboard, see the link printed when we started the Dask client.

In [ ]:
features.to_dask_dataframe().reset_index(drop=True).to_parquet("dp2_vsx_features", write_index=False)

In [ ]:
client.close()

## Non-DRH side: load the shared features

From here on, we use only the saved files. They contain public VSX data, DiaObject IDs and the features, so anybody can work with them, including non-DRHs. Let's see how many eclipsing binaries and LPVs we have.

In [ ]:
shared_features = pd.read_parquet("dp2_vsx_features")
shared_features["is_eclipsing"].value_counts()

In [ ]:
shared_features.head()

## Non-DRH side: train a classifier

Finally, let's train a random forest classifier with default parameters to separate eclipsing binaries from LPVs.

First, a little feature engineering: the Otsu mean difference and standard deviations are in flux units, so they scale with the variability amplitude. We normalize them by the interpercentile range between the 5th and 95th percentiles of the flux, and drop the range itself, so all the features are dimensionless.

In [ ]:
engineered = shared_features.copy()
for band in BANDS:
    scale = engineered.pop(f"inter_percentile_range_5_{band}")
    for name in ["otsu_mean_diff", "otsu_std_lower", "otsu_std_upper"]:
        engineered[f"{name}_{band}"] /= scale

feature_names = [name for name in extractor.names if not name.startswith("inter_percentile_range")]

We balance the classes by randomly subsampling the larger one, and use 80% of the objects for training and 20% for testing.

In [ ]:
# Balance the classes by random subsampling
n_per_class = engineered["is_eclipsing"].value_counts().min()
balanced = engineered.groupby("is_eclipsing").sample(n=n_per_class, random_state=42)

X = balanced[feature_names]
y = balanced["is_eclipsing"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train, y_train)
print(classification_report(y_test, classifier.predict(X_test), target_names=["LPV", "eclipsing"]))

Which features are the most useful for the classifier?

In [ ]:
importances = pd.Series(classifier.feature_importances_, index=feature_names).sort_values()

fig, ax = plt.subplots(figsize=(7, 9))
importances.plot.barh(ax=ax)
ax.set_xlabel("Feature importance")
plt.tight_layout()
plt.show()

## Problem: similarity search

**Your task:** a non-DRH colleague gives you the coordinates of an object and asks you to find DP2 objects similar to it.

**Steps:**
1. Find the DP2 DiaObject at these coordinates, e.g. with a small `lsdb.ConeSearch`.
2. Embed the DP2 light curves in a small sky region around it, e.g. with the ATCAT model from `light_curve.embed`.
3. Find the nearest neighbours of the given object in the embedding space.
4. Share back their DiaObject IDs, cosine distances, and embeddings.

One possible solution is in `Notebook_3b_SimilaritySearch.ipynb`.

## Citing

If you use the tools or the data from this notebook in your research, please cite the following papers.

<details>
<summary>For LSDB and HATS cite: Caplar et al. (2025), arXiv:2501.02103</summary>

```bibtex
% ADS bibcode: 2025arXiv250102103C
@ARTICLE{Caplar_LSDB_2025,
       author = {{Caplar}, Neven and {Beebe}, Wilson and {Branton}, Doug and {Campos}, Sandro and {Connolly}, Andrew and {DeLucchi}, Melissa and {Jones}, Derek and {Juric}, Mario and {Kubica}, Jeremy and {Malanchev}, Konstantin and {Mandelbaum}, Rachel and {McGuire}, Sean},
        title = "{Using LSDB to enable large-scale catalog distribution, cross-matching, and analytics}",
      journal = {arXiv e-prints},
     keywords = {Instrumentation and Methods for Astrophysics},
         year = 2025,
        month = jan,
          eid = {arXiv:2501.02103},
        pages = {arXiv:2501.02103},
          doi = {10.48550/arXiv.2501.02103},
archivePrefix = {arXiv},
       eprint = {2501.02103},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2025arXiv250102103C},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For Rubin catalogs in HATS cite: Malanchev et al. (2025), arXiv:2506.23955</summary>

```bibtex
% ADS bibcode: 2025arXiv250623955M
@ARTICLE{Malanchev_DP1_2025,
       author = {{Malanchev}, Konstantin and {DeLucchi}, Melissa and {Caplar}, Neven and {Malz}, Alex I. and {Alexov}, Anastasia and {Aubourg}, Eric and {E Bauer}, Amanda and {Beebe}, Wilson and {Bellm}, Eric C. and {Blum}, Robert David and {Branton}, Doug and {Campos}, Sandro and {Calabrese}, Daniel and {Carlin}, Jeffrey L. and {Choi}, Yumi and {Connolly}, Andrew and {Dai}, Mi and {Daly}, Philip N. and {Daruich}, Felipe and {Daubard}, Guillaume and {Delgado}, Francisco and {Drass}, Holger and {Fonseca Alvarez}, Gloria and {Gangler}, Emmanuel and {Guy}, Leanne P. and {Ivezi{\'c}}, {\v{Z}}eljko and {Jammes}, Fabrice and {Jannuzi}, Buell T. and {Jenness}, Tim and {Jimenez}, David and {Jones}, Derek T. and {Juri{\'c}}, Mario and {Kahn}, Steven M. and {Kang}, Yijung and {Kannawadi}, Arun and {Kubica}, Jeremy and {Le Guillou}, Laurent and {Lynn}, Olivia and {Mandelbaum}, Rachel and {Moniez}, Marc and {Morrison}, Christopher B. and {McGuire}, Sean and {Ordenes}, Ian E. and {Patterson}, Maria T. and {Plazas Malag{\'o}n}, Andr{\'e}s A. and {Qiu}, Yongqiang and {Quint}, Bruno and {Radeka}, Veljko and {van Reeven}, Wouter and {Riot}, Vincent J. and {S{\'a}nchez}, Bruno O. and {Sanmartim}, David and {Sebag}, Jacques and {Shugart}, Alysha and {Sullivan}, Ian S. and {Taranu}, Dan S. and {Urbach}, Elana K. and {Wood-Vasey}, W.~M. and {Zhang}, Tianqing},
        title = "{Variability-finding in Rubin Data Preview 1 with LSDB}",
      journal = {arXiv e-prints},
     keywords = {Instrumentation and Methods for Astrophysics, Solar and Stellar Astrophysics},
         year = 2025,
        month = jun,
          eid = {arXiv:2506.23955},
        pages = {arXiv:2506.23955},
          doi = {10.48550/arXiv.2506.23955},
archivePrefix = {arXiv},
       eprint = {2506.23955},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2025arXiv250623955M},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For Rubin DP2 data cite: Rubin Observatory (2026), RTN-115, doi:10.71929/rubin/3377440; DP2 dataset, doi:10.71929/rubin/3382528; DP2 DiaObject catalog, doi:10.71929/rubin/3382534; DP2 ForcedSourceOnDiaObject catalog, doi:10.71929/rubin/3382536</summary>

```bibtex
% ADS bibcode: 2026rubn.rept....2V
@TECHREPORT{Rubin_DP2_2026,
       author = {{Vera C. Rubin Observatory Team}},
        title = "{RTN-115: The Vera C. Rubin Observatory Data Preview 2}",
  institution = {NSF-DOE Vera C. Rubin Observatory},
     keywords = {79 ASTRONOMY AND ASTROPHYSICS},
         year = 2026,
 howpublished = {NSF-DOE Vera C. Rubin Observatory Technical Report},
          doi = {10.71929/RUBIN/3377440},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2026rubn.rept....2V},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}

% ADS bibcode: 2026lsst.data...42N
@dataset{Rubin_DP2Data_2026,
       author = {{NSF-DOE Vera C. Rubin Observatory}},
        title = "{Legacy Survey of Space and Time Data Preview 2}",
 howpublished = {NSF-DOE Vera C. Rubin Observatory Data Set},
         year = 2026,
        month = jan,
          doi = {10.71929/RUBIN/3382528},
    publisher = {Rubin Obs},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2026lsst.data...42N},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}

% ADS bibcode: 2026lsst.data...32N
@dataset{Rubin_DP2DiaObject_2026,
       author = {{NSF-DOE Vera C. Rubin Observatory}},
        title = "{Legacy Survey of Space and Time Data Preview 2: DiaObject searchable catalog}",
 howpublished = {NSF-DOE Vera C. Rubin Observatory Data Set},
         year = 2026,
        month = jan,
          doi = {10.71929/RUBIN/3382534},
    publisher = {Rubin Obs},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2026lsst.data...32N},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}

% ADS bibcode: 2026lsst.data...44N
@dataset{Rubin_DP2ForcedSourceOnDiaObject_2026,
       author = {{NSF-DOE Vera C. Rubin Observatory}},
        title = "{Legacy Survey of Space and Time Data Preview 2: ForcedSourceOnDiaObject searchable catalog}",
 howpublished = {NSF-DOE Vera C. Rubin Observatory Data Set},
         year = 2026,
        month = jan,
          doi = {10.71929/RUBIN/3382536},
    publisher = {Rubin Obs},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2026lsst.data...44N},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For VSX cite: Watson et al. (2006), SASS, 25, 47</summary>

```bibtex
% ADS bibcode: 2006SASS...25...47W
@ARTICLE{Watson_VSX_2006,
       author = {{Watson}, C.~L. and {Henden}, A.~A. and {Price}, A.},
        title = "{The International Variable Star Index (VSX)}",
      journal = {Society for Astronomical Sciences Annual Symposium},
         year = 2006,
        month = may,
       volume = {25},
        pages = {47},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2006SASS...25...47W},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For light-curve cite: Malanchev et al. (2021), MNRAS, 502, 5147</summary>

```bibtex
% ADS bibcode: 2021MNRAS.502.5147M
@ARTICLE{Malanchev_ZTF_2021,
       author = {{Malanchev}, K.~L. and {Pruzhinskaya}, M.~V. and {Korolev}, V.~S. and {Aleo}, P.~D. and {Kornilov}, M.~V. and {Ishida}, E.~E.~O. and {Krushinsky}, V.~V. and {Mondon}, F. and {Sreejith}, S. and {Volnova}, A.~A. and {Belinski}, A.~A. and {Dodin}, A.~V. and {Tatarnikov}, A.~M. and {Zheltoukhov}, S.~G. and {(The SNAD Team)}},
        title = "{Anomaly detection in the Zwicky Transient Facility DR3}",
      journal = {\mnras},
     keywords = {methods: data analysis, astronomical data bases: miscellaneous, stars: variables: general, Astrophysics - Instrumentation and Methods for Astrophysics, Astrophysics - Solar and Stellar Astrophysics},
         year = 2021,
        month = apr,
       volume = {502},
       number = {4},
        pages = {5147-5175},
          doi = {10.1093/mnras/stab316},
archivePrefix = {arXiv},
       eprint = {2012.01419},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2021MNRAS.502.5147M},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For BeyondNStd cite: D'Isanto et al. (2016), MNRAS, 457, 3119</summary>

```bibtex
% ADS bibcode: 2016MNRAS.457.3119D
@ARTICLE{DIsanto_Transients_2016,
       author = {{D'Isanto}, A. and {Cavuoti}, S. and {Brescia}, M. and {Donalek}, C. and {Longo}, G. and {Riccio}, G. and {Djorgovski}, S.~G.},
        title = "{An analysis of feature relevance in the classification of astronomical transients with machine learning methods}",
      journal = {\mnras},
     keywords = {methods: data analysis, novae, cataclysmic variables, supernovae: general, stars: variables: general, stars: variables: RR Lyrae, Astrophysics - Instrumentation and Methods for Astrophysics},
         year = 2016,
        month = apr,
       volume = {457},
       number = {3},
        pages = {3119-3132},
          doi = {10.1093/mnras/stw157},
archivePrefix = {arXiv},
       eprint = {1601.03931},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2016MNRAS.457.3119D},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For Chi2Pvar cite: Kim et al. (2014), A&A, 566, A43</summary>

```bibtex
% ADS bibcode: 2014A&A...566A..43K
@ARTICLE{Kim_EPOCH_2014,
       author = {{Kim}, Dae-Won and {Protopapas}, Pavlos and {Bailer-Jones}, Coryn A.~L. and {Byun}, Yong-Ik and {Chang}, Seo-Won and {Marquette}, Jean-Baptiste and {Shin}, Min-Su},
        title = "{The EPOCH Project. I. Periodic variable stars in the EROS-2 LMC database}",
      journal = {\aap},
     keywords = {stars: variables: general, Magellanic Clouds, methods: data analysis, Astrophysics - Instrumentation and Methods for Astrophysics, Astrophysics - Solar and Stellar Astrophysics},
         year = 2014,
        month = jun,
       volume = {566},
          eid = {A43},
        pages = {A43},
          doi = {10.1051/0004-6361/201323252},
archivePrefix = {arXiv},
       eprint = {1403.6131},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2014A&A...566A..43K},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For OtsuSplit cite: Otsu (1979), IEEE TSMC, 9, 62; Lavrukhina et al. (2023), RNAAS, 7, 199</summary>

```bibtex
% ADS bibcode: 1979ITSMC...9...62O
@ARTICLE{Otsu_Threshold_1979,
       author = {{Otsu}, Nobuyuki},
        title = "{A Threshold Selection Method from Gray-Level Histograms}",
      journal = {IEEE Transactions on Systems, Man, and Cybernetics},
     keywords = {Histograms, Marine vehicles, Radar tracking, Least squares approximation, Surveillance, Target tracking, Gaussian distribution, Displays, Q measurement, Sea measurements},
         year = 1979,
        month = jan,
       volume = {9},
       number = {1},
        pages = {62-66},
          doi = {10.1109/TSMC.1979.4310076},
       adsurl = {https://ui.adsabs.harvard.edu/abs/1979ITSMC...9...62O},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}

% ADS bibcode: 2023RNAAS...7..199L
@ARTICLE{Lavrukhina_Plateau_2023,
       author = {{Lavrukhina}, Anastasia and {Malanchev}, Konstantin and {Kornilov}, Matwey V.},
        title = "{Automatic Detection of Plateau Phases in Light Curves of Variable Stars}",
      journal = {Research Notes of the American Astronomical Society},
     keywords = {Time series analysis, 1916, Astrophysics - Instrumentation and Methods for Astrophysics},
         year = 2023,
        month = sep,
       volume = {7},
       number = {9},
          eid = {199},
        pages = {199},
          doi = {10.3847/2515-5172/acfd2d},
archivePrefix = {arXiv},
       eprint = {2309.10174},
 primaryClass = {astro-ph.IM},
       adsurl = {https://ui.adsabs.harvard.edu/abs/2023RNAAS...7..199L},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>

<details>
<summary>For StetsonK cite: Stetson (1996), PASP, 108, 851</summary>

```bibtex
% ADS bibcode: 1996PASP..108..851S
@ARTICLE{Stetson_Cepheid_1996,
       author = {{Stetson}, Peter B.},
        title = "{On the Automatic Determination of Light-Curve Parameters for Cepheid Variables}",
      journal = {\pasp},
     keywords = {CEPHEIDS, METHODS: NUMERICAL},
         year = 1996,
        month = oct,
       volume = {108},
        pages = {851},
          doi = {10.1086/133808},
       adsurl = {https://ui.adsabs.harvard.edu/abs/1996PASP..108..851S},
      adsnote = {Provided by the SAO/NASA Astrophysics Data System}
}
```

</details>